# Predictive Maintenance Prototype

**Author:** Simon Htet  
**Context:** DairyPlus — Tetra Pak Filling Machine Fleet

---

## Overview

The Smart Factory Platform collects real-time telemetry from 23 Tetra Pak filling machines: running hours, temperature readings, vibration levels, and CIP cycle status. This notebook prototypes a **Random Forest classifier** that uses those same signals to predict the probability of machine breakdown before it occurs.

The goal is **predictive maintenance** — shift from reactive (fix it after it breaks) to proactive (flag it before it breaks), reducing unplanned downtime and its impact on OEE.

### Features used
| Feature | Source | Why it matters |
|---|---|---|
| `Running_Hour` | PLC odometer counter | Wear accumulates with runtime |
| `Heat_C` | Temperature sensor | Abnormal heat = friction or lubrication failure |
| `Vibration` | Vibration sensor | Imbalance or bearing wear shows up here first |

### Status
This is a **prototype on simulated data**. The pipeline architecture already collects all three signals from live PLCs. Next step is wiring this model to the live `pipeline/` data feed and scheduling inference via the existing SQL Server job agent.


In [ ]:
# Install dependencies (if running outside the project environment)
# !pip install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 1. Simulated Machine Telemetry

Simulates a representative sample of machine records. In production, this data comes from the SQL Server tables populated by `pipeline/collector.py`.

In [ ]:
data = {
    'Running_Hour': [500, 1200, 2000, 2500, 2800, 3200, 3700, 4000, 4200, 4500],
    'Heat_C':       [45,   52,   50,   55,   53,   56,   57,   53,   54,   58],
    'Vibration':    [0.2,  0.5,  0.4,  0.3,  0.45, 0.6,  0.5,  0.6,  0.65, 0.7],
    'Breakdown':    [0,    1,    0,    1,    0,    1,    1,    1,    1,    1]
}

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} records | Breakdown rate: {df['Breakdown'].mean():.0%}")
df

## 2. Train / Test Split

In [ ]:
X = df[['Running_Hour', 'Heat_C', 'Vibration']]
y = df['Breakdown']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

## 3. Model Training

Random Forest is well-suited here: it handles small datasets without overfitting as aggressively as a single decision tree, and `feature_importances_` gives a direct read on which sensor signals matter most — useful for prioritising sensor calibration.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Feature importance — which signal matters most?
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature importance:")
print(importance.to_string())

## 4. Evaluation

In [ ]:
predictions = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, predictions, target_names=['OK', 'Breakdown']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

## 5. Breakdown Risk Scoring

Rather than a binary prediction, `predict_proba` gives a continuous risk score (0–100%). This is more actionable for maintenance scheduling — a machine at 78% risk gets flagged for inspection before it hits 100%.

In [ ]:
def breakdown_risk(running_hour, heat_c, vibration):
    """Return breakdown risk as a percentage for a given machine reading."""
    sample = pd.DataFrame({
        'Running_Hour': [running_hour],
        'Heat_C': [heat_c],
        'Vibration': [vibration]
    })
    prob = model.predict_proba(sample)[0][1]
    return round(prob * 100, 1)

# Example: machine showing high hours + abnormal heat
risk = breakdown_risk(running_hour=6000, heat_c=70, vibration=0.6)
print(f"Estimated Breakdown Risk: {risk}%")

# Threshold-based alert (would feed into maintenance work order system)
ALERT_THRESHOLD = 70
status = 'MAINTENANCE REQUIRED' if risk >= ALERT_THRESHOLD else 'OK'
print(f"Maintenance Alert: {status}")

## 6. Next Steps (Production Path)

| Step | What | How |
|---|---|---|
| 1 | Replace simulated data | Query live telemetry from SQL Server via `pyodbc` |
| 2 | Retrain on real labels | Use historical breakdown records from maintenance log table |
| 3 | Schedule inference | SQL Server Agent job calling this script on each shift |
| 4 | Surface alerts | Write risk scores back to SQL Server → Power BI alert tile |
| 5 | Expand features | Add CIP cycle count, product changeover frequency, ambient temperature |

The pipeline infrastructure for steps 1, 3, and 4 already exists in `pipeline/`.